# AISE3010 Assignment 4

## Analysis and Optimization of CUDA Matrix Multiplication

This report analyzes the CUDA matrix multiplication program provided in the uploaded notebook and presents an improved implementation that satisfies the required conditions. The assignment focused on understanding how CUDA can be used for optimization, identifying the weaknesses of an existing solution, and producing a corrected version that is more reliable and scalable.


## Required Conditions

The assignment states that the program should be able to run under one of the following three conditions:

- Condition (i): `A in R^(500x500)`, `B in R^(500x400)`, `N > 100`
- Condition (ii): `A in R^(50x20)`, `B in R^(20x50)`, `N > 5000`
- Condition (iii): `A in R^(6x4000)`, `B in R^(4000x9)`, `N > 1000`

These conditions show that the solution must support rectangular matrix multiplication and must remain valid under repeated execution.


## Overview of the Original Program

The original notebook contains a CPU implementation, a GPU kernel, unified memory allocation, and runtime measurements intended to compare CPU and GPU execution. It demonstrates the main idea behind CUDA programming by allocating memory, launching a kernel, synchronizing the device, and freeing memory at the end. This gives the code some value as an introductory example.

At the same time, the implementation does not match the actual assignment conditions. The program tests only three square-style cases based on `N = 150`, `300`, and `500`, while the assignment requires rectangular matrices with fixed dimensions. Because of that mismatch, the code does not provide a valid answer to the problem as stated.


## Advantages of the Original Program

One advantage of the original program is that it follows the basic CUDA workflow clearly. It separates the CPU and GPU matrix multiplication functions, which makes it easier to compare both approaches. It also attempts to measure execution time, which shows that performance evaluation was part of the intended solution. As a starting point, the code is useful because it introduces the structure of a CUDA-based matrix multiplication program.


## Disadvantages of the Original Program

The main weakness of the original code is that it does not follow the actual problem definition. The assignment requires rectangular matrix multiplication, but the program is written as though the problem is square and controlled by a single dimension `N`. This creates incorrect indexing when the number of columns is not equal to `N`.

Another issue is that the result matrix is accumulated using `+=` without being initialized to zero first. This can lead to incorrect output because the program may add values onto uninitialized memory. The CPU and GPU versions also do not run on the same input values because the matrices are randomized again before the GPU run, which makes the comparison unreliable. In addition, the original implementation does not numerically verify that the CPU and GPU outputs match, so there is no strong evidence that the result is correct.

Finally, the timing approach is weak for GPU evaluation. The use of `clock()` is not the best method for measuring GPU kernel performance, and the kernel configuration is fixed rather than based on the true matrix dimensions.


## Why the Original Program Fails

The original program fails because it is built around assumptions that do not hold for the three official conditions. The assignment uses rectangular matrices, but the uploaded code indexes arrays using `N` in places where the actual column count should be used. This means the logic breaks down as soon as the dimensions differ from a square layout. On top of that, the output matrix is never initialized before accumulation begins, and the CPU and GPU runs do not use the same data. Because of these issues, the program is not reliable for correctness, and its timing results are not enough to support a meaningful analysis.


## Improved Approach

To correct these issues, the improved implementation uses the real matrix dimensions directly instead of forcing the problem into a square format. A standard 2D CUDA kernel is used so that each thread computes one output element. This makes the thread mapping much clearer and allows the grid configuration to scale naturally with the matrix shape. The improved version also initializes memory properly, uses the same input data for both CPU and GPU computation, and checks correctness by comparing the two outputs.

For profiling, CUDA event timing is used for the GPU execution rather than relying only on `clock()`. This makes the performance measurements more appropriate for CUDA code and provides a better basis for comparing results.


In [1]:
!nvidia-smi
!nvcc --version

Tue Apr  7 01:22:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
%%writefile assignment4_fixed.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <chrono>
#include <cuda_runtime.h>

#define CHECK_CUDA(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(EXIT_FAILURE); \
    } \
} while (0)

__global__ void matmulKernel(const double *A, const double *B, double *C, int rowsA, int colsA, int colsB) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < rowsA && col < colsB) {
        double sum = 0.0;
        for (int k = 0; k < colsA; ++k) {
            sum += A[row * colsA + k] * B[k * colsB + col];
        }
        C[row * colsB + col] = sum;
    }
}

void matmulCPU(const double *A, const double *B, double *C, int rowsA, int colsA, int colsB) {
    for (int i = 0; i < rowsA; ++i) {
        for (int j = 0; j < colsB; ++j) {
            double sum = 0.0;
            for (int k = 0; k < colsA; ++k) {
                sum += A[i * colsA + k] * B[k * colsB + j];
            }
            C[i * colsB + j] = sum;
        }
    }
}

void fillMatrix(double *M, int rows, int cols) {
    for (int i = 0; i < rows * cols; ++i) {
        M[i] = 2.0 + (rand() / (double) RAND_MAX) * 2.0;
    }
}

double maxAbsDiff(const double *X, const double *Y, int size) {
    double maxDiff = 0.0;
    for (int i = 0; i < size; ++i) {
        double diff = fabs(X[i] - Y[i]);
        if (diff > maxDiff) maxDiff = diff;
    }
    return maxDiff;
}

void runCase(const char *label, int rowsA, int colsA, int rowsB, int colsB, int repeatCount) {
    int sizeA = rowsA * colsA;
    int sizeB = rowsB * colsB;
    int sizeC = rowsA * colsB;

    double *A = (double*) malloc(sizeA * sizeof(double));
    double *B = (double*) malloc(sizeB * sizeof(double));
    double *C_cpu = (double*) malloc(sizeC * sizeof(double));
    double *C_gpu = (double*) malloc(sizeC * sizeof(double));
    double *d_A, *d_B, *d_C;

    CHECK_CUDA(cudaMalloc((void**)&d_A, sizeA * sizeof(double)));
    CHECK_CUDA(cudaMalloc((void**)&d_B, sizeB * sizeof(double)));
    CHECK_CUDA(cudaMalloc((void**)&d_C, sizeC * sizeof(double)));

    fillMatrix(A, rowsA, colsA);
    fillMatrix(B, rowsB, colsB);

    auto cpuStart = std::chrono::high_resolution_clock::now();
    for (int r = 0; r < repeatCount; ++r) matmulCPU(A, B, C_cpu, rowsA, colsA, colsB);
    auto cpuEnd = std::chrono::high_resolution_clock::now();
    double cpuMs = std::chrono::duration<double, std::milli>(cpuEnd - cpuStart).count();

    CHECK_CUDA(cudaMemcpy(d_A, A, sizeA * sizeof(double), cudaMemcpyHostToDevice));
    CHECK_CUDA(cudaMemcpy(d_B, B, sizeB * sizeof(double), cudaMemcpyHostToDevice));

    dim3 block(16, 16);
    dim3 grid((colsB + block.x - 1) / block.x, (rowsA + block.y - 1) / block.y);
    cudaEvent_t start, stop;
    CHECK_CUDA(cudaEventCreate(&start));
    CHECK_CUDA(cudaEventCreate(&stop));

    CHECK_CUDA(cudaEventRecord(start));
    for (int r = 0; r < repeatCount; ++r) matmulKernel<<<grid, block>>>(d_A, d_B, d_C, rowsA, colsA, colsB);
    CHECK_CUDA(cudaEventRecord(stop));
    CHECK_CUDA(cudaEventSynchronize(stop));
    CHECK_CUDA(cudaGetLastError());

    float gpuMs = 0.0f;
    CHECK_CUDA(cudaEventElapsedTime(&gpuMs, start, stop));
    CHECK_CUDA(cudaMemcpy(C_gpu, d_C, sizeC * sizeof(double), cudaMemcpyDeviceToHost));
    double diff = maxAbsDiff(C_cpu, C_gpu, sizeC);

    printf("\n%s\n", label);
    printf("A: %d x %d, B: %d x %d, repeats: %d\n", rowsA, colsA, rowsB, colsB, repeatCount);
    printf("CPU time: %.3f ms\n", cpuMs);
    printf("GPU kernel time: %.3f ms\n", gpuMs);
    printf("Max absolute difference: %.12f\n", diff);

    CHECK_CUDA(cudaEventDestroy(start));
    CHECK_CUDA(cudaEventDestroy(stop));
    CHECK_CUDA(cudaFree(d_A));
    CHECK_CUDA(cudaFree(d_B));
    CHECK_CUDA(cudaFree(d_C));
    free(A); free(B); free(C_cpu); free(C_gpu);
}

int main() {
    srand(7);
    printf("AISE3010 Assignment 4: Corrected CUDA Matrix Multiplication\n");
    runCase("Condition (i)", 500, 500, 500, 400, 101);
    runCase("Condition (ii)", 50, 20, 20, 50, 5001);
    runCase("Condition (iii)", 6, 4000, 4000, 9, 1001);
    return 0;
}


Writing assignment4_fixed.cu


In [3]:
!nvcc -O2 assignment4_fixed.cu -o assignment4_fixed

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./assignment4_fixed

AISE3010 Assignment 4: Corrected CUDA Matrix Multiplication

Condition (i)
A: 500 x 500, B: 500 x 400, repeats: 101
CPU time: 14915.716 ms
GPU kernel time: 111.750 ms
Max absolute difference: 0.000000000003

Condition (ii)
A: 50 x 20, B: 20 x 50, repeats: 5001
CPU time: 181.768 ms
GPU kernel time: 25.709 ms
Max absolute difference: 0.000000000000

Condition (iii)
A: 6 x 4000, B: 4000 x 9, repeats: 1001
CPU time: 301.874 ms
GPU kernel time: 301.883 ms
Max absolute difference: 0.000000000015


## Profiling and Performance Discussion

The original program attempted to compare CPU and GPU runtime, but the comparison was not fully reliable because the CPU and GPU versions did not use the same input matrices and the program did not verify that both outputs matched. In the improved implementation, the same matrices are used for both computations, and the output is checked numerically using the maximum absolute difference. This gives a more complete evaluation because it considers correctness together with performance.

During testing in Google Colab, `nvprof` may not be supported on newer GPUs with compute capability 8.0 or higher. In that case, the CUDA event timing built into the program still provides valid profiling evidence for GPU execution.


## Results

| Condition | Matrix Dimensions | Repeat Count | CPU Time | GPU Time | Max Difference |
|---|---|---:|---:|---:|---:|
| (i) | 500x500 times 500x400 | 101 | 14915.716 ms | 111.750 ms | 0.000000000003 |
| (ii) | 50x20 times 20x50 | 5001 | 181.768 ms | 25.709 ms | 0.000000000000 |
| (iii) | 6x4000 times 4000x9 | 1001 | 301.874 ms | 301.883 ms | 0.000000000015 |

The corrected implementation is designed to work directly with the three required matrix configurations rather than using a square-matrix assumption. Because each case uses the real matrix dimensions, the solution is more general and aligned with the assignment. The numerical difference between CPU and GPU outputs should remain very small, which confirms that the GPU implementation is producing the correct result.


## Bonus: CUDA Optimization of Assignment 1

The Assignment 1 code was a manually implemented multilayer perceptron for classification and regression on the car evaluation dataset. After reviewing the MATLAB files, the part of the code most suitable for CUDA acceleration is the repeated matrix-heavy computation inside forward propagation, backpropagation, and gradient updates. Even though the full Assignment 1 project was written in MATLAB, the underlying operations are dominated by dense linear algebra, especially expressions of the form `W * A + b`, gradient products, and repeated mini-batch updates.

For that reason, the bonus was approached as a CUDA C acceleration of the core matrix computation that dominates the training loop, rather than as a full line-by-line rewrite of the entire MATLAB project. This is a more realistic optimization strategy because the main performance benefit of CUDA comes from accelerating the repeated matrix multiplications that appear throughout the network training process.


In [5]:
%%writefile assignment1_bonus_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <chrono>
#include <cuda_runtime.h>

#define CHECK_CUDA(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(EXIT_FAILURE); \
    } \
} while (0)

__global__ void denseMatmulKernel(const double *W, const double *X, double *Y, int outDim, int inDim, int batch) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < outDim && col < batch) {
        double sum = 0.0;
        for (int k = 0; k < inDim; ++k) {
            sum += W[row * inDim + k] * X[k * batch + col];
        }
        Y[row * batch + col] = sum;
    }
}

void denseMatmulCPU(const double *W, const double *X, double *Y, int outDim, int inDim, int batch) {
    for (int i = 0; i < outDim; ++i) {
        for (int j = 0; j < batch; ++j) {
            double sum = 0.0;
            for (int k = 0; k < inDim; ++k) {
                sum += W[i * inDim + k] * X[k * batch + j];
            }
            Y[i * batch + j] = sum;
        }
    }
}

void fillArray(double *arr, int n) {
    for (int i = 0; i < n; ++i) arr[i] = (rand() / (double) RAND_MAX) * 2.0 - 1.0;
}

double maxAbsDiff(const double *a, const double *b, int n) {
    double maxDiff = 0.0;
    for (int i = 0; i < n; ++i) {
        double diff = fabs(a[i] - b[i]);
        if (diff > maxDiff) maxDiff = diff;
    }
    return maxDiff;
}

void runLayerCase(const char *label, int outDim, int inDim, int batch, int repeats) {
    int sizeW = outDim * inDim;
    int sizeX = inDim * batch;
    int sizeY = outDim * batch;

    double *W = (double*) malloc(sizeW * sizeof(double));
    double *X = (double*) malloc(sizeX * sizeof(double));
    double *Ycpu = (double*) malloc(sizeY * sizeof(double));
    double *Ygpu = (double*) malloc(sizeY * sizeof(double));
    double *dW, *dX, *dY;

    CHECK_CUDA(cudaMalloc((void**)&dW, sizeW * sizeof(double)));
    CHECK_CUDA(cudaMalloc((void**)&dX, sizeX * sizeof(double)));
    CHECK_CUDA(cudaMalloc((void**)&dY, sizeY * sizeof(double)));

    fillArray(W, sizeW);
    fillArray(X, sizeX);

    auto cpuStart = std::chrono::high_resolution_clock::now();
    for (int r = 0; r < repeats; ++r) denseMatmulCPU(W, X, Ycpu, outDim, inDim, batch);
    auto cpuEnd = std::chrono::high_resolution_clock::now();
    double cpuMs = std::chrono::duration<double, std::milli>(cpuEnd - cpuStart).count();

    CHECK_CUDA(cudaMemcpy(dW, W, sizeW * sizeof(double), cudaMemcpyHostToDevice));
    CHECK_CUDA(cudaMemcpy(dX, X, sizeX * sizeof(double), cudaMemcpyHostToDevice));

    dim3 block(16, 16);
    dim3 grid((batch + block.x - 1) / block.x, (outDim + block.y - 1) / block.y);
    cudaEvent_t start, stop;
    CHECK_CUDA(cudaEventCreate(&start));
    CHECK_CUDA(cudaEventCreate(&stop));

    CHECK_CUDA(cudaEventRecord(start));
    for (int r = 0; r < repeats; ++r) denseMatmulKernel<<<grid, block>>>(dW, dX, dY, outDim, inDim, batch);
    CHECK_CUDA(cudaEventRecord(stop));
    CHECK_CUDA(cudaEventSynchronize(stop));
    CHECK_CUDA(cudaGetLastError());

    float gpuMs = 0.0f;
    CHECK_CUDA(cudaEventElapsedTime(&gpuMs, start, stop));
    CHECK_CUDA(cudaMemcpy(Ygpu, dY, sizeY * sizeof(double), cudaMemcpyDeviceToHost));

    printf("\n%s\n", label);
    printf("W: %d x %d, X: %d x %d, repeats: %d\n", outDim, inDim, inDim, batch, repeats);
    printf("CPU time: %.3f ms\n", cpuMs);
    printf("GPU time: %.3f ms\n", gpuMs);
    printf("Max absolute difference: %.12f\n", maxAbsDiff(Ycpu, Ygpu, sizeY));

    CHECK_CUDA(cudaEventDestroy(start));
    CHECK_CUDA(cudaEventDestroy(stop));
    CHECK_CUDA(cudaFree(dW));
    CHECK_CUDA(cudaFree(dX));
    CHECK_CUDA(cudaFree(dY));
    free(W); free(X); free(Ycpu); free(Ygpu);
}

int main() {
    srand(42);
    printf("Assignment 1 Bonus CUDA Prototype\n");
    runLayerCase("Classification layer-style workload", 48, 21, 64, 20000);
    runLayerCase("Regression layer-style workload", 32, 17, 64, 20000);
    return 0;
}


Writing assignment1_bonus_cuda.cu


In [6]:
!nvcc -O2 assignment1_bonus_cuda.cu -o assignment1_bonus_cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [7]:
!./assignment1_bonus_cuda

Assignment 1 Bonus CUDA Prototype

Classification layer-style workload
W: 48 x 21, X: 21 x 64, repeats: 20000
CPU time: 982.080 ms
GPU time: 126.866 ms
Max absolute difference: 0.000000000000

Regression layer-style workload
W: 32 x 17, X: 17 x 64, repeats: 20000
CPU time: 514.121 ms
GPU time: 73.439 ms
Max absolute difference: 0.000000000000


## Bonus Discussion

The CUDA bonus prototype above focuses on the dense matrix multiplication pattern that appears repeatedly in the Assignment 1 neural network code. In the MATLAB implementation, those operations occur during forward propagation and gradient computation. By moving that matrix-heavy portion to CUDA C, it becomes possible to test whether the GPU can produce similar numerical results while reducing runtime for repeated workloads.

This bonus implementation should be understood as a targeted optimization of the most computationally expensive part of the Assignment 1 pipeline rather than a full replacement of the entire MATLAB project. That is still a meaningful extension, because the repeated matrix products are the part of the training loop that benefits most directly from GPU parallelism. If the GPU time is lower than the CPU time while the maximum absolute difference remains very small, then the CUDA version can be considered to produce similar results with improved performance.


## Bonus Results

| Bonus Case | CPU Time(ms) | GPU Time(ms) | Max Difference | Notes |
|---|---:|---:|---:|---|
| Classification layer-style workload | 982.080 | 126.866 | 0.000000000000 | The GPU was more efficient |
| Regression layer-style workload | 514.121 | 73.439 | 0.000000000000 | The GPU was more efficient |

To connect these results back to Assignment 1, the original report recorded a classification training time of about `0.870 s` on a single CPU and a regression training time of about `0.393 s`. The CUDA prototype is not a full end-to-end rewrite of that MATLAB project, so the comparison should be framed carefully. The correct claim is that CUDA accelerated the core matrix computation that dominates the training loop, which suggests that a full low-level CUDA implementation of the network would likely reduce runtime further for repeated training workloads.


## Conclusion

This assignment showed that CUDA optimization is not only about moving a computation to the GPU, but also about making sure the implementation is correct, scalable, and consistent with the actual problem definition. The original code demonstrated the basic structure of CUDA programming, but it did not satisfy the rectangular matrix conditions required in the assignment and contained several correctness and measurement issues. After redesigning the solution to use proper indexing, dimension-aware kernel launches, consistent inputs, and correctness checks, the final implementation became a more complete and reliable answer to the problem.

The bonus portion also showed that CUDA can be applied to the matrix-heavy parts of the Assignment 1 neural network workflow. Even though the prototype focused on the dense linear algebra rather than the full MATLAB training pipeline, the result still demonstrates how GPU acceleration can be used to improve the performance of the most computationally expensive operations while maintaining numerically similar output.
